In [7]:
#Task1
from queue import PriorityQueue

university = {
    'S': ['A', 'B', 'C'],
    'A': ['D', 'E'],
    'B': ['F', 'G'],
    'C': ['H'],
    'H': ['I', 'J'],
    'I': ['K', 'L', 'M'],
    'D': [], 'E': [],
    'F': [], 'G': [],
    'J': [], 'K': [],
    'L': [], 'M': []
}

# h(n): estimated walking distance from each node to goal 'I'
heuristic = {
    'S': 10, 'A': 8, 'B': 9, 'C': 5,
    'H': 3,  'I': 0, 'J': 6, 'K': 7,
    'D': 12, 'E': 11, 'F': 13, 'G': 14,
    'L': 8,  'M': 5
}

def best_first_search(graph, heuristic, start, goal):
    OPEN   = PriorityQueue()   
    CLOSED = set()               
    came_from = {start: None}  

    OPEN.put((heuristic[start], start))

    while not OPEN.empty():
        h, node = OPEN.get()

        if node in CLOSED:
            continue

        CLOSED.add(node)
        print(node + " -> ", end='')

        if node == goal:
            # Reconstruct path
            path = []
            while node is not None:
                path.append(node)
                node = came_from[node]
            path.reverse()
            print("Goal Reached")
            print("Path:", " -> ".join(path))
            return

        for neighbour in graph[node]:
            if neighbour not in CLOSED:
                came_from[neighbour] = node
                OPEN.put((heuristic[neighbour], neighbour))

    print("Goal Not Found")

best_first_search(university, heuristic, 'S', 'I')

S -> C -> H -> I -> Goal Reached
Path: S -> C -> H -> I


In [5]:
#Task2

# 0 = empty, 1 = wall, 'O' = object
grid = [
    [0,   0,   0,   1,   0,   0,   0],
    [0,   1,   0,   0,   0,   1,   0],
    [0,   0,  'O',  1,   0,   0,  'O'],
    [1,   0,   0,   0,   1,   0,   0],
    [0,   0,   1,  'O',  0,   0,   0],
    [0,   0,   0,   0,   0,   1,  'O'],
]

ROWS, COLS = len(grid), len(grid[0])
START = (0, 0)

def heuristic(pos, goal):
    return abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])

def get_neighbours(r, c):
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and grid[nr][nc] != 1:
            yield (nr, nc)

def greedy_bfs(start, goal):
    frontier = [(heuristic(start, goal), start)]
    came_from = {start: None}
    visited   = set()

    while frontier:
        frontier.sort(key=lambda x: x[0])      # sort by h(n)
        _, current = frontier.pop(0)

        if current in visited:
            continue
        visited.add(current)

        print(f"  Visiting {current}", end=" -> ")

        if current == goal:
            # Reconstruct path
            path, node = [], current
            while node is not None:
                path.append(node)
                node = came_from[node]
            path.reverse()
            return path

        for nb in get_neighbours(*current):
            if nb not in visited:
                came_from[nb] = current
                frontier.append((heuristic(nb, goal), nb))

    return None

def find_closest(pos, objects):
    return min(objects, key=lambda obj: heuristic(pos, obj))

def robot_collect(grid, start):
    objects = [
        (r, c)
        for r in range(ROWS)
        for c in range(COLS)
        if grid[r][c] == 'O'
    ]

    pos            = start
    visited_cells  = {start}
    collection_order = []

    print(f"Robot starts at {start}")
    print(f"Objects to collect: {objects}\n")

    while objects:
        target = find_closest(pos, objects)
        print(f" Nearest object: {target}  (h={heuristic(pos, target)}) ")

        path = greedy_bfs(pos, target)

        if path is None:
            print(f"No path to {target}, skipping.\n")
            objects.remove(target)
            continue

        for cell in path:
            visited_cells.add(cell)

        print(f"\n  Path: {path}")

        objects.remove(target)
        collection_order.append(target)
        r, c = target
        grid[r][c] = 'X'   
        pos = target
        print(f"  Collected! Remaining objects: {len(objects)}\n")

    print(f"All objects collected!")
    print(f"Collection order: {collection_order}")

robot_collect(grid, START)

Robot starts at (0, 0)
Objects to collect: [(2, 2), (2, 6), (4, 3), (5, 6)]

 Nearest object: (2, 2)  (h=4) 
  Visiting (0, 0) ->   Visiting (1, 0) ->   Visiting (2, 0) ->   Visiting (2, 1) ->   Visiting (2, 2) -> 
  Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)]
  Collected! Remaining objects: 3

 Nearest object: (4, 3)  (h=3) 
  Visiting (2, 2) ->   Visiting (3, 2) ->   Visiting (3, 3) ->   Visiting (4, 3) -> 
  Path: [(2, 2), (3, 2), (3, 3), (4, 3)]
  Collected! Remaining objects: 2

 Nearest object: (5, 6)  (h=4) 
  Visiting (4, 3) ->   Visiting (5, 3) ->   Visiting (5, 4) ->   Visiting (4, 4) ->   Visiting (4, 5) ->   Visiting (4, 6) ->   Visiting (5, 6) -> 
  Path: [(4, 3), (5, 3), (5, 4), (4, 4), (4, 5), (4, 6), (5, 6)]
  Collected! Remaining objects: 1

 Nearest object: (2, 6)  (h=3) 
  Visiting (5, 6) ->   Visiting (4, 6) ->   Visiting (3, 6) ->   Visiting (2, 6) -> 
  Path: [(5, 6), (4, 6), (3, 6), (2, 6)]
  Collected! Remaining objects: 0

All objects collected!
Collection o

In [6]:
from queue import PriorityQueue

campus = {
    'Gate':     {'LibBlock': 4,  'SciBlock': 6,  'MedBlock': 3},
    'LibBlock': {'AdminBlock': 5, 'CafeBlock': 2},
    'SciBlock': {'CafeBlock': 7,  'LabBlock': 8},
    'MedBlock': {'LabBlock': 4,  'ExitA': 9},
    'AdminBlock':{'ExitB': 6},
    'CafeBlock': {'ExitB': 3,   'LabBlock': 5},
    'LabBlock':  {'ExitA': 2,   'ExitB': 7},
    'ExitA':     {},
    'ExitB':     {},
}

heuristic = {
    'Gate':      12,
    'LibBlock':  9,
    'SciBlock':  10,
    'MedBlock':  7,
    'AdminBlock':5,
    'CafeBlock': 6,
    'LabBlock':  2,
    'ExitA':     0,
    'ExitB':     3,
}

BLOCKED_EDGES = {
    ('MedBlock', 'ExitA'),   
    ('SciBlock', 'LabBlock'), 
}

UNSAFE_NODES = {'AdminBlock'}  

def a_star(graph, heuristic, start, goal,
           blocked_edges=set(), unsafe_nodes=set()):

    OPEN   = [(heuristic[start], start)] 
    CLOSED = set()

    g_cost    = {start: 0}
    came_from = {start: None}

    while OPEN:
        OPEN.sort(key=lambda x: x[0])          
        current_f, current = OPEN.pop(0)

        if current in CLOSED:
            continue

        CLOSED.add(current)
        print(f"  Expanding: {current}  [g={g_cost[current]}, h={heuristic[current]}, f={current_f}]")

        if current == goal:
            # Reconstruct path
            path, node = [], current
            total_time = g_cost[current]
            while node is not None:
                path.append(node)
                node = came_from[node]
            path.reverse()
            print(f"\nEvacuation route found!")
            print(f"Path      : {' -> '.join(path)}")
            print(f"Total time: {total_time} mins")
            return path

        for neighbour, travel_time in graph[current].items():

            #Safety checks 
            if neighbour in unsafe_nodes:
                print(f"Skipping {neighbour} — unsafe zone")
                continue

            if (current, neighbour) in blocked_edges:
                print(f"Skipping {current}->{neighbour} — blocked route")
                continue

            if neighbour in CLOSED:
                continue

            new_g = g_cost[current] + travel_time
            new_f = new_g + heuristic[neighbour]

            if neighbour not in g_cost or new_g < g_cost[neighbour]:
                g_cost[neighbour]    = new_g
                came_from[neighbour] = current
                OPEN.append((new_f, neighbour))

    print("No safe evacuation route found!")
    return None

print("Emergency Evacuation A*\n")
a_star(
    graph         = campus,
    heuristic     = heuristic,
    start         = 'Gate',
    goal          = 'ExitA',
    blocked_edges = BLOCKED_EDGES,
    unsafe_nodes  = UNSAFE_NODES
)

Emergency Evacuation A*

  Expanding: Gate  [g=0, h=12, f=12]
  Expanding: MedBlock  [g=3, h=7, f=10]
Skipping MedBlock->ExitA — blocked route
  Expanding: LabBlock  [g=7, h=2, f=9]
  Expanding: ExitA  [g=9, h=0, f=9]

Evacuation route found!
Path      : Gate -> MedBlock -> LabBlock -> ExitA
Total time: 9 mins


['Gate', 'MedBlock', 'LabBlock', 'ExitA']